# CS2 Deathmatch Bot - YOLOv8n Training

Run this notebook on Google Colab (free GPU).

**Steps:**
1. Upload your dataset zip
2. Run all cells
3. Download the exported ONNX model

In [ ]:
# Install dependencies
!pip install ultralytics onnx onnxruntime -q

In [ ]:
# Upload your dataset
# Option A: Upload zip via Colab file browser (left sidebar -> Files -> Upload)
# Option B: Use Google Drive

import os
from google.colab import files

print("Upload your Roboflow YOLOv8 export zip file:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

In [ ]:
# Unzip dataset
!mkdir -p /content/dataset
!unzip -o "{zip_name}" -d /content/dataset
!ls /content/dataset/

In [ ]:
# Create a proper data.yaml with correct paths
# Since Roboflow only gave us train, we'll split 90/10 for train/val

import shutil
import random

train_img_dir = '/content/dataset/train/images'
train_lbl_dir = '/content/dataset/train/labels'
val_img_dir = '/content/dataset/valid/images'
val_lbl_dir = '/content/dataset/valid/labels'

os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

# Check if valid already exists with data
if len(os.listdir(val_img_dir)) == 0:
    # Split 10% for validation
    all_images = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    random.seed(42)
    random.shuffle(all_images)
    val_count = max(1, len(all_images) // 10)
    val_images = all_images[:val_count]

    for img_name in val_images:
        lbl_name = os.path.splitext(img_name)[0] + '.txt'
        shutil.move(os.path.join(train_img_dir, img_name), os.path.join(val_img_dir, img_name))
        lbl_src = os.path.join(train_lbl_dir, lbl_name)
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(val_lbl_dir, lbl_name))

    print(f'Split: {len(all_images) - val_count} train, {val_count} val')
else:
    print(f'Validation set already exists with {len(os.listdir(val_img_dir))} images')

# Write corrected data.yaml
data_yaml = """train: /content/dataset/train/images
val: /content/dataset/valid/images

nc: 2
names: ['ct_player', 'head']
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml written')
print(f'Train images: {len(os.listdir(train_img_dir))}')
print(f'Val images: {len(os.listdir(val_img_dir))}')

In [ ]:
# Train YOLOv8n
from ultralytics import YOLO

# Load pretrained YOLOv8 nano (fastest, good enough for our use case)
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # Early stopping if no improvement for 20 epochs
    device=0,             # GPU
    workers=2,
    project='/content/runs',
    name='cs2_bot',
    exist_ok=True,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    flipud=0.0,           # No vertical flip (players don't appear upside down)
    fliplr=0.5,           # Horizontal flip is fine
    mosaic=0.8,
    mixup=0.1,
)

print('Training complete!')

In [ ]:
# Show training results
from IPython.display import Image, display

# Results plots
display(Image('/content/runs/cs2_bot/results.png', width=800))
print()
display(Image('/content/runs/cs2_bot/confusion_matrix.png', width=600))

In [ ]:
# Validate on validation set
best_model = YOLO('/content/runs/cs2_bot/weights/best.pt')
metrics = best_model.val(data='/content/dataset/data.yaml')

print(f"\nmAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

In [ ]:
# Export to ONNX for use with onnxruntime-directml (AMD GPU)
best_model = YOLO('/content/runs/cs2_bot/weights/best.pt')

onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=17,
    dynamic=False,
)

print(f'\nONNX model exported to: {onnx_path}')
print(f'Size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')

In [ ]:
# Download the ONNX model
# Save this as models/cs2_yolov8n.onnx in your project
from google.colab import files

files.download('/content/runs/cs2_bot/weights/best.onnx')
print('\nDone! Save this file as: models/cs2_yolov8n.onnx in your project folder')